# Aegis-Scryer RAG - PDF extraction and chunking
Books: Mindfulness in Plain English, Permission to Feel, The 48 Laws of Power, The Art of War, Ultralearning.
Output: `scryer_knowledge_chunks.json`, ready for embedding + Weaviate upload.

**IMPORTANT:** raw PDF text is never uploaded to Weaviate, only the resulting chunks.

Pipeline: extract lines -> strip running headers / page numbers / watermarks -> detect chapters (per book) ->
reflow hard-wrapped lines into paragraphs (de-hyphenate) -> split into sentences -> pack sentences into ~400-token chunks
with whole-sentence overlap.

In [1]:
import pymupdf
import re
import json
import unicodedata
from collections import Counter
from pathlib import Path

## Config

In [2]:
# heading = which chapter detector to use (see "Chapter detection" below)
# exclude_chapters = detected chapter titles dropped from the output
BOOKS = [
    {
        "path": "knowledge/Mindfulness_in_plain_english.pdf",
        "domain": "mindfulness",
        "book": "Mindfulness in Plain English",
        "author": "Bhante Gunaratana",
        "page_range": [13, 73],
        "heading": "mindfulness",
    },
    {
        "path": "knowledge/Permission_to_feel.pdf",
        "domain": "emotions",
        "book": "Permission to Feel",
        "author": "Marc Brackett",
        "page_range": [12, 213],
        "heading": "permission",
    },
    {
        "path": "knowledge/The_48_laws_of_power.pdf",
        "domain": "life_strategy",
        "book": "The 48 Laws of Power",
        "author": "Robert Greene",
        # p3-12 = summary table of contents (repeats every law's Judgment), Preface starts p13,
        # Law 48 ends p580, bibliography starts p581.
        "page_range": [3, 580],
        "heading": "laws48",
        "exclude_chapters": ["Contents (summary of the laws)"],  # remove this line to keep the summary
    },
    {
        "path": "knowledge/The_Art_Of_War.pdf",
        "domain": "life_strategy",
        "book": "The Art of War",
        "author": "Sun Tzu",
        "page_range": [4, 65],  # p3 is the contents page
        "heading": "artofwar",
    },
    {
        "path": "knowledge/Ultralearning.pdf",
        "domain": "learning_strategies",
        "book": "Ultralearning",
        "author": "Scott H Young",
        "page_range": [12, 219],
        "heading": "ultralearning",
    },
    {
        "path": "knowledge/The_Prince.pdf",
        "domain": "life_strategy",
        "book": "The Prince",
        "author": "Nicollo Machiavelli",
        "page_range": [2, 131],
        "heading": "prince",
    },
]

In [3]:
CHUNK_TARGET_TOKENS = 400
CHUNK_OVERLAP_TOKENS = 70
CHARS_PER_TOKEN = 4  # rough heuristic for English prose
CHUNK_TARGET_CHARS = CHUNK_TARGET_TOKENS * CHARS_PER_TOKEN      # 1600
CHUNK_OVERLAP_CHARS = CHUNK_OVERLAP_TOKENS * CHARS_PER_TOKEN    # 280 (target); whole sentences, so capped below
CHUNK_OVERLAP_MAX_CHARS = 400
MIN_CHUNK_CHARS = 250           # a shorter trailing chunk is merged into the previous one

WATERMARKS = ["acquired at wisdompubs.org"]

## Step 1 - extract pages as clean lines
Lines are NFKC-normalised (fixes ligatures such as `ﬁ` -> `fi`). Running headers/footers are detected
automatically: any line that shows up in the first/last 2 lines of >= 4 pages is a header.

In [4]:
def extract_pages(pdf_path, page_range):
    doc = pymupdf.open(pdf_path)
    start, end = page_range
    pages = []
    for i in range(start - 1, min(end, len(doc))):
        raw = doc[i].get_text("text")
        lines = [unicodedata.normalize("NFKC", l).strip() for l in raw.split("\n")]
        pages.append({"page": i + 1, "lines": [l for l in lines if l]})
    doc.close()
    return pages


# chapter markers repeat on many pages but are headings, not running headers
CHAPTER_MARKER = re.compile(r"(C\s*H\s*A\s*P\s*T\s*E\s*R|Chapter\s+\w+|PART\s+\w+|LAW\s*\d+)")


def find_running_headers(pages, min_pages=4):
    counts = Counter()
    for p in pages:
        edge = set(p["lines"][:2] + p["lines"][-2:])
        for l in edge:
            if not l.isdigit() and not CHAPTER_MARKER.fullmatch(l):
                counts[l.lower()] += 1
    return {l for l, n in counts.items() if n >= min_pages}

## Step 2 - chapter detection (one detector per book)
Each detector gets a page's lines and returns `[(line_index, chapter_title, n_lines_consumed)]`.
The consumed lines (the heading itself) are removed from the body text. Detectors run on lines *before*
page-number stripping because some books use a bare number as the chapter marker.

In [5]:
def is_title_like(line):
    words = line.split()
    if not words:
        return False
    caps = sum(1 for w in words if w[0].isupper())
    return caps / len(words) >= 0.6
 
 


In [6]:
def detect_prince(lines, st):
    # Each page opens with running-header lines first ('The Prince' / page no. / 'Nicolo Machiavelli'),
    # THEN 'CHAPTER I' - it is not lines[0]. Scan the first few lines like detect_mindfulness does.
    # ['CHAPTER I', 'How Many Kinds Of Principalities There', 'Are, And By What Means They Are', 'Acquired', <body>]
    # title spans 1-4 lines in Title Case; body starts once a line stops being title-like.
    for i, l in enumerate(lines[:5]):
        if re.fullmatch(r"CHAPTER [IVXL]+", l):
            title, k = [], i + 1
            while k < len(lines) and k <= i + 4 and is_title_like(lines[k]) \
                    and not (len(lines[k].split()) == 1 and lines[k].isupper()):  # skip drop-cap remnants (a lone ALL-CAPS word, e.g. 'HE'/'OMING' from a split 'THE'/'COMING') - real titles here are Title Case, never one all-caps word
                title.append(lines[k])
                k += 1
            return [(i, f"{l}: {' '.join(title)}", k - i)]
    return []


In [7]:
def _is_garbled(line):
    return sum(ord(c) < 32 for c in line) > 0.3 * max(len(line), 1)


def detect_mindfulness(lines, st):
    # ['1', 'C H A P T E R', '1', 'Meditation: Why Bother?', ...]
    for i, l in enumerate(lines[:6]):
        if re.fullmatch(r"C\s*H\s*A\s*P\s*T\s*E\s*R", l):
            j = i + 1
            num = ""
            if j < len(lines) and re.fullmatch(r"\d(\s?\d)?", lines[j]):
                num, j = lines[j].replace(" ", ""), j + 1
            if j < len(lines):
                return [(i, f"Chapter {num}: {lines[j]}" if num else lines[j], j - i + 1)]
    return []


def detect_permission(lines, st):
    # ['2', 'Emotions Are Information', 'SO, HOW ARE YOU FEELING?', ...]; the chapter number must be sequential.
    # 'PART TWO' + title pages are consumed without starting a chapter.
    if lines and re.fullmatch(r"PART (ONE|TWO|THREE)", lines[0]):
        k = 1
        while k < len(lines) and k < 3 and lines[k].isupper():
            k += 1
        return [(0, None, k)]
    if len(lines) >= 3 and lines[0].isdigit() and int(lines[0]) == st.get("n", 0) + 1   and not lines[1].isdigit() and len(lines[1]) < 60:
        st["n"] = int(lines[0])
        title, n = lines[1], 2
        if len(lines[2]) < 20 and not lines[2].isupper():     # title wrapped onto a 2nd line ("... Preschool to" / "College")
            title, n = f"{title} {lines[2]}", 3
        return [(0, f"Chapter {lines[0]}: {title}", n)]
    return []


def detect_ultralearning(lines, st):
    # ['Chapter II', 'Why Ultralearning Matters', <body...>]  title = following short lines
    if lines and re.fullmatch(r"Chapter [IVXL]+", lines[0]):
        title, k = [], 1
        while k < len(lines) and k <= 3 and len(lines[k]) < 60 and not lines[k].endswith((".", ",")):
            title.append(lines[k]); k += 1
        return [(0, f"{lines[0]}: {' '.join(title)}".strip(": "), k)]
    return []


ART_OF_WAR_TITLES = {
    "I": "Laying Plans", "II": "Waging War", "III": "Attack by Stratagem", "IV": "Tactical Dispositions",
    "V": "Energy", "VI": "Weak Points and Strong", "VII": "Manoeuvring", "VIII": "Variation in Tactics",
    "IX": "The Army on the March", "X": "Terrain", "XI": "The Nine Situations",
    "XII": "The Attack by Fire", "XIII": "The Use of Spies",
}


def detect_artofwar(lines, st):
    # ['II', <garbled-font title>, '1. Sun Tzu said: ...']  -> title comes from the known list
    if lines and lines[0] in ART_OF_WAR_TITLES:
        n = 2 if len(lines) > 1 and _is_garbled(lines[1]) else 1
        return [(0, f"Chapter {lines[0]}: {ART_OF_WAR_TITLES[lines[0]]}", n)]
    return []


def detect_laws48(lines, st):
    # 'LAW 1' + all-caps title (may wrap) + 'JUDGMENT'.  Laws must be sequential: cross references
    # like "see Law 24" and stray matches are ignored.
    found = []
    for i, l in enumerate(lines):
        if l == "PREFACE" and not st.get("preface"):
            st["preface"] = True
            found.append((i, "Preface", 1))
            continue
        m = re.fullmatch(r"LAW\s*(\d)\s?(\d?)", l)
        if m and int(m.group(1) + m.group(2)) == st.get("n", 0) + 1:
            n = int(m.group(1) + m.group(2))
            title, k = [], i + 1
            while k < len(lines) and len(title) < 3 and lines[k].isupper() and lines[k] != "JUDGMENT":
                title.append(lines[k]); k += 1
            st["n"] = n
            found.append((i, f"Law {n}: {' '.join(title)}", k - i))
    return found


DETECTORS = {
    "mindfulness": detect_mindfulness,
    "permission": detect_permission,
    "ultralearning": detect_ultralearning,
    "artofwar": detect_artofwar,
    "laws48": detect_laws48,
}

## Step 3 - flatten pages into one stream of (page, line), split at chapter headings
Also removes page numbers (digit-only lines at the top/bottom of a page), watermarks, garbled footers, and rejoins
drop caps (`M` / `editation is...` -> `Meditation is...`).

In [8]:
def build_chapters(book_config):
    pages = extract_pages(book_config["path"], book_config["page_range"])
    headers = find_running_headers(pages)
    detect = DETECTORS[book_config["heading"]]
    state = {}

    # before the first heading we are in the front matter
    front = "Contents (summary of the laws)" if book_config["heading"] == "laws48" else "Introduction"
    chapters = [{"title": front, "lines": []}]

    for p in pages:
        lines = p["lines"]          # headers are dropped below, AFTER detection (a chapter title can equal the running header)
        heads = {i: (title, n) for i, title, n in detect(lines, state)}
        consumed = {i + k for i, (_, n) in heads.items() for k in range(n)}
        n_lines = len(lines)
        for i, l in enumerate(lines):
            if i in heads and heads[i][0]:
                chapters.append({"title": heads[i][0], "lines": [], "start_page": p["page"]})
                continue
            if i in consumed:
                continue
            at_edge = i < 2 or i >= n_lines - 2
            if l.lower() in headers:
                continue
            if any(w in l.lower() for w in WATERMARKS):
                continue
            if _is_garbled(l):
                continue
            if at_edge and l.isdigit():
                continue
            chapters[-1]["lines"].append((p["page"], l))

    for c in chapters:
        c["lines"] = fix_drop_caps(c["lines"])
    return [c for c in chapters if c["lines"]]


def fix_drop_caps(lines):
    out, i = [], 0
    while i < len(lines):
        page, text = lines[i]
        if len(text) == 1 and i + 1 < len(lines):
            nxt = lines[i + 1][1]
            if text.isalpha() and text.isupper() and nxt[:1].islower():
                out.append((page, text + nxt)); i += 2
                continue
            i += 1  # stray single character
            continue
        out.append((page, text)); i += 1
    return out

## Step 4 - reflow into paragraphs, then sentences
PyMuPDF returns one `\n` per visual line and almost no blank lines, so paragraph breaks are inferred: a line that is
noticeably shorter than the page's text width and ends a sentence closes the paragraph (and numbered items in
The Art of War start a new one). Hyphenated line breaks (`high-` / `ly`) are rejoined.

In [9]:
def reflow_paragraphs(lines):
    widths = sorted(len(t) for _, t in lines)
    wrap = widths[int(len(widths) * 0.9)] if widths else 80
    paras, cur = [], None  # cur = [page, text]
    for page, text in lines:
        if cur is None:
            cur = [page, text]
            continue
        prev = cur[1]
        starts_item = re.match(r"\d+\.\s", text) is not None and prev.endswith((".", "!", "?", ":", ";", "\u201d"))
        short_end = len(prev) < 0.72 * wrap and prev.endswith((".", "!", "?", ":", "\u201d", '"', ")"))
        if starts_item or short_end:
            paras.append(tuple(cur))
            cur = [page, text]
        elif prev.endswith("-") and text[:1].islower():
            cur[1] = prev[:-1] + text          # de-hyphenate
        elif prev.endswith("\u2014"):
            cur[1] = prev + text
        else:
            cur[1] = prev + " " + text
    if cur:
        paras.append(tuple(cur))
    return paras


SENT_SPLIT = re.compile(r"(?:(?<=[.!?\u2026])|(?<=[.!?\u2026][\u201d\"\u2019')]))\s+(?=[A-Z\u201c\"\u2018(\d])")


def split_sentences(paras):
    out = []
    for page, text in paras:
        for s in SENT_SPLIT.split(text):
            s = re.sub(r"\s+", " ", s).strip()
            if s:
                out.append((s, page))
    return out


def split_long(sentence, limit=CHUNK_TARGET_CHARS):
    if len(sentence) <= limit:
        return [sentence]
    words, pieces, cur = sentence.split(" "), [], ""
    for w in words:
        if cur and len(cur) + len(w) + 1 > limit:
            pieces.append(cur); cur = w
        else:
            cur = f"{cur} {w}".strip()
    if cur:
        pieces.append(cur)
    return pieces

## Step 5 - pack sentences into chunks with whole-sentence overlap

In [10]:
def pack_chunks(sentences):
    chunks, cur, cur_len, n_ov = [], [], 0, 0
    for s, page in sentences:
        for piece in split_long(s):
            if cur and cur_len + len(piece) + 1 > CHUNK_TARGET_CHARS:
                chunks.append(cur)
                ov, n = [], 0
                for x in reversed(cur):                       # carry trailing whole sentences
                    if n + len(x[0]) + 1 > min(CHUNK_OVERLAP_MAX_CHARS, CHUNK_OVERLAP_CHARS * 1.4):
                        break
                    ov.insert(0, x); n += len(x[0]) + 1
                cur, cur_len, n_ov = ov, n, len(ov)
            cur.append((piece, page)); cur_len += len(piece) + 1
    if cur and len(cur) > n_ov:
        if chunks and cur_len < MIN_CHUNK_CHARS:
            chunks[-1] = chunks[-1] + cur[n_ov:]              # fold a tiny tail into the previous chunk
        else:
            chunks.append(cur)
    return chunks


def build_chunk_records(book_config):
    exclude = set(book_config.get("exclude_chapters", []))
    records = []
    for chapter in build_chapters(book_config):
        if chapter["title"] in exclude:
            continue
        sentences = split_sentences(reflow_paragraphs(chapter["lines"]))
        for idx, chunk in enumerate(pack_chunks(sentences)):
            records.append({
                "domain": book_config["domain"],
                "book": book_config["book"],
                "author": book_config["author"],
                "chapter": chapter["title"],
                "chunk_index": idx,
                "start_page": chunk[0][1],
                "source": "book",
                "text": " ".join(s for s, _ in chunk),
            })
    return records

In [11]:
DETECTORS["prince"] = detect_prince

## Step 6 - run all books and save

In [12]:
all_records = []
for book_config in BOOKS:
    recs = build_chunk_records(book_config)
    print(f"{book_config['book']}: {len(recs)} chunks")
    all_records.extend(recs)

output_path = Path("scryer_knowledge_chunks.json")
output_path.write_text(json.dumps(all_records, ensure_ascii=False, indent=2))
print(f"Saved {len(all_records)} total chunks to {output_path}")

Mindfulness in Plain English: 93 chunks


Permission to Feel: 366 chunks


The 48 Laws of Power: 1073 chunks
The Art of War: 51 chunks


Ultralearning: 361 chunks
The Prince: 150 chunks
Saved 2094 total chunks to scryer_knowledge_chunks.json


In [13]:
prince_records = [r for r in all_records if r["book"] == "The Prince"]
print(f"The Prince: {len(prince_records)} chunks, {len(set(r['chapter'] for r in prince_records))} chapters")

The Prince: 150 chunks, 27 chapters


In [14]:
print(prince_records[1]["chapter"])
print(prince_records[1]["text"][:250])

CHAPTER I: How Many Kinds Of Principalities There Are, And By What Means They Are Acquired
LL STATES, all powers, that have held and hold rule over men have been and are either republics or principalities. Principalities are either hereditary, in which the family has been long established; or they are new. The new are either entirely new, 


## Step 7 - sanity checks (inspect before embedding)

In [15]:
import statistics
for book in BOOKS:
    recs = [r for r in all_records if r["book"] == book["book"]]
    lens = [len(r["text"]) for r in recs]
    chapters = list(dict.fromkeys(r["chapter"] for r in recs))
    print(f"\n{book['book']}: {len(recs)} chunks, chars min/median/max = {min(lens)}/{int(statistics.median(lens))}/{max(lens)}, {len(chapters)} chapters")
    print("  chapters:", chapters[:4], "...", chapters[-2:])

bad = {
    "watermark": lambda t: "wisdompubs" in t.lower(),
    "hyphen-linebreak": lambda t: "-\n" in t or "\n" in t,
    "control chars": lambda t: any(ord(c) < 32 for c in t),
    "starts lowercase": lambda t: t[:1].islower(),
}
print()
for name, fn in bad.items():
    hits = [r for r in all_records if fn(r["text"])]
    print(f"{name}: {len(hits)}")


Mindfulness in Plain English: 93 chunks, chars min/median/max = 488/1556/1599, 5 chapters
  chapters: ['Chapter 1: Meditation: Why Bother?', 'Chapter 2: What Meditation Isn’t', 'Chapter 3: What Meditation Is', 'Chapter 10: Dealing with Problems'] ... ['Chapter 10: Dealing with Problems', 'Chapter 13: Mindfulness (Sati)']

Permission to Feel: 366 chunks, chars min/median/max = 357/1539/1599, 11 chapters
  chapters: ['Chapter 1: Permission to Feel', 'Chapter 2: Emotions Are Information', 'Chapter 3: How to Become an Emotion Scientist', 'Chapter 4: R: Recognizing Emotion'] ... ['Chapter 10: Emotions at School: From Preschool to College', 'Chapter 11: Emotions at Work']

The 48 Laws of Power: 1073 chunks, chars min/median/max = 219/1532/1599, 49 chapters
  chapters: ['Law 1: NEVER OUTSHINE THE MASTER', 'Law 2: NEVER PUT TOO MUCH TRUST IN FRIENDS, LEARN HOW TO USE ENEMIES', 'Law 3: CONCEAL YOUR INTENTIONS', 'Law 4: ALWAYS SAY LESS THAN NECESSARY'] ... ['Law 47: DO NOT GO PAST THE MARK YOU 

control chars: 0
starts lowercase: 0


In [16]:
# eyeball a few chunks per book
import random
random.seed(1)
for book in BOOKS:
    r = random.choice([r for r in all_records if r["book"] == book["book"]])
    print("---", r["book"], "|", r["chapter"], "| idx", r["chunk_index"], "| p", r["start_page"])
    print(r["text"][:400])

--- Mindfulness in Plain English | Chapter 2: What Meditation Isn’t | idx 1 | p 23
They make statements about meditation that sound like general laws but are actually highly specific procedures exclusive to that particular system of practice. Worse yet is the panoply of complex theories and interpretations available, often at odds with one another. The result is a real mess: an enormous jumble of conflicting opinions accompanied by a mass of extraneous data. This book is specifi
--- Permission to Feel | Chapter 10: Emotions at School: From Preschool to College | idx 7 | p 169
It reminded us that what teachers really teach is themselves—their contagious passion for their subjects and students. It reminded us that children learn from people they love, and that love in this context means willing the good of another, and offering active care for the whole person.” Emotion as a part of the learning experience doesn’t simply mean that students and teachers will bare their so
--- The 48 Laws 

In [17]:
import os

In [18]:
from dotenv import load_dotenv, find_dotenv


In [19]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams,
    Distance,
    PayloadSchemaType,
)

In [20]:
load_dotenv(find_dotenv(usecwd=True)) 

True

In [21]:
client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    timeout=120,  # default 5s is too short for uploading batches of 1536-dim vectors
)

In [22]:
# Create the Scryer_Knowledge collection
# Single unnamed dense vector: 1536 dims (text-embedding-3-small), cosine distance.
# on_disk keeps vectors off RAM, which matters on the free tier (1GB RAM).

In [23]:
COLLECTION_NAME = "Scryer_Knowledge"

In [24]:
if client.collection_exists(COLLECTION_NAME):
    print(f"{COLLECTION_NAME} already exists - skipping creation")
else:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=1536,
            distance=Distance.COSINE,
            on_disk=True,
        ),
    )
    print(f"Created {COLLECTION_NAME}")
 

Scryer_Knowledge already exists - skipping creation


In [25]:
# Payload index on 'domain' so we can filter retrieval per domain
# (mindfulness | emotions | life_strategy | learning_strategies)

In [26]:
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="domain",
    field_schema=PayloadSchemaType.KEYWORD,
)
print("Created keyword index on 'domain'")
 
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="book",
    field_schema=PayloadSchemaType.KEYWORD,
)
print("Created keyword index on 'book'")

Created keyword index on 'domain'


Created keyword index on 'book'


In [27]:
import uuid
from openai import OpenAI
from qdrant_client.models import PointStruct, Filter, FieldCondition, MatchValue

In [28]:
 
info = client.get_collection(COLLECTION_NAME)
print(info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=2094 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=True, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, pre

In [29]:
openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment
EMBED_MODEL = "text-embedding-3-small"  # 1536 dims, matches the collection
BATCH = 50


In [30]:
existing = client.count(COLLECTION_NAME).count
if existing >= len(all_records):  # ids are deterministic, so an interrupted upload is safely resumed by re-running
    print(f"{COLLECTION_NAME} already has {existing} points - skipping upload")
else:
    for i in range(0, len(all_records), BATCH):
        batch = all_records[i:i + BATCH]
        resp = openai_client.embeddings.create(model=EMBED_MODEL, input=[r["text"] for r in batch])
        client.upsert(
            collection_name=COLLECTION_NAME,
            wait=True,
            points=[
                PointStruct(
                    id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{r['book']}|{r['chapter']}|{r['chunk_index']}")),
                    vector=e.embedding,
                    payload=r,  # includes text + domain/book/author/chapter/chunk_index/start_page/source
                )
                for r, e in zip(batch, resp.data)
            ],
        )
    print("uploaded", client.count(COLLECTION_NAME).count, "points")

Scryer_Knowledge already has 2094 points - skipping upload


## Test retrieval

In [31]:
q = openai_client.embeddings.create(model=EMBED_MODEL, input="how do I stop being controlled by anger?").data[0].embedding
hits = client.query_points(
    COLLECTION_NAME, query=q, limit=3,
    query_filter=Filter(must=[FieldCondition(key="domain", match=MatchValue(value="emotions"))]),
).points
for h in hits:
    print(f"{h.score:.3f} | {h.payload['book']} | {h.payload['chapter']} | p{h.payload['start_page']}\n  {h.payload['text'][:150]}...")

0.496 | Permission to Feel | Chapter 9: Emotions at Home | p149
  “However,” I say. “This doesn’t get you off the hook. It doesn’t mean you can keep yelling and screaming and punishing and stressing everybody out. It...
0.494 | Permission to Feel | Chapter 1: Permission to Feel | p12
  When a parent or teacher is faced with what might appear to be an angry child, often the first impulse is to threaten discipline—if you don’t stop yel...
0.492 | Permission to Feel | Chapter 1: Permission to Feel | p12
  We need to access that information and then figure out what it’s telling us. That way we can make the most informed decisions. That’s a major challeng...


### Adding a book later
`The Prince` (Machiavelli) was added 2026-09-27: a `prince` chapter detector above, one entry in `BOOKS`, re-run everything through **Step 6**. The upload cell above is idempotent (it skips once Qdrant already holds `len(all_records)` points), so adding one more book only embeds the new chunks, not the whole library — as long as its `book`/`chapter` names are final before the first upload (they are part of the deterministic id). A wrong `book` value or missed chapter (as happened here) is fixed with `client.set_payload` / delete-and-reupload for just the affected points, not a full re-embed.